In [1]:
import os
import sys
from dotenv import load_dotenv
from pathlib import Path
import datetime

import pandas as pd
import statsmodels.api as sm
import getFamaFrenchFactors as gff

In [2]:
# environment variables
env_path = Path("/home/jovyan/.env")
load_dotenv(env_path)

# mounted data directory in container
data_path = Path("/home/jovyan/data/data-main")

print("cwd:", Path.cwd())
print("data_path:", data_path)
print("exists:", data_path.exists())

cwd: /home/jovyan
data_path: /home/jovyan/data/data-main
exists: True


In [4]:
# finance/data/data-main/4_multi_factor_data.csv
data = pd.read_csv(data_path / "4_multi_factor_data.csv", parse_dates=True).sort_index()
data['Date'] = pd.to_datetime(data['Date'])
df = data.set_index('Date')
company_list = ['Tesla', 'GM']
df.columns = company_list
df.head()

,Tesla,GM
Date,,
2012-01-03,1.87200,16.1959
2012-01-04,1.84733,16.2746
2012-01-05,1.80800,17.0591
2012-01-06,1.79400,17.6354
2012-01-09,1.81667,17.5757


### 月次リターンと因子データの整形
個別株の月次リターンは次式で計算します。

$$
R_{i,t} = \frac{P_{i,t}}{P_{i,t-1}} - 1
$$

ここで $P_{i,t}$ は銘柄 $i$ の月末株価です。Fama-French データとの結合後、超過収益率は

$$
R^{e}_{i,t} = R_{i,t} - RF_t
$$

で定義し、$RF_t$ は同月の無リスク金利です。以降の回帰では、この超過収益率を被説明変数として使います。

In [6]:
Fama_French_3 = gff.famaFrench3Factor(frequency='m')
# 月次データ取得
Fama_French_3.rename(columns={'date_ff_factors': 'Date'}, inplace=True)
Fama_French_3.set_index('Date', inplace=True)
Returns = df.resample('M').last().pct_change().dropna()
# 月次データへ変換
Fama_French_data = Fama_French_3.merge(Returns, on='Date')

/tmp/ipykernel_107/3346542066.py:5: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  Returns = df.resample('M').last().pct_change().dropna()


### Fama-French 3因子モデル
このセルでは Fama-French 3因子モデル

$$
R^{e}_{i,t} = \alpha_i + \beta_i (Mkt{-}RF)_t + s_i SMB_t + h_i HML_t + \varepsilon_{i,t}
$$

を最小二乗法で推定します。

- $R^{e}_{i,t}$: 銘柄 $i$ の超過収益率
- $(Mkt{-}RF)_t$: 市場超過収益率
- $SMB_t$: 小型株要因
- $HML_t$: バリュー要因
- $\alpha_i$: 異常収益率
- $\varepsilon_{i,t}$: 誤差項

`sm.add_constant(X)` は定数項 $\alpha_i$ を回帰式に含めるための処理です。

In [7]:
X = Fama_French_data[['Mkt-RF', 'SMB', 'HML']]
# 重回帰分析の説明変数定義
y = Fama_French_data['Tesla'] - Fama_French_data['RF']
# 被説明変数定義
X = sm.add_constant(X)
Fama_French_Model = sm.OLS(y, X).fit()
print(Fama_French_Model.summary().tables[1])
print('AdjR2: %.4f'% Fama_French_Model.rsquared_adj)

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0286      0.015      1.918      0.057      -0.001       0.058
Mkt-RF         1.8455      0.348      5.303      0.000       1.157       2.534
SMB            0.4180      0.613      0.682      0.497      -0.795       1.631
HML           -1.0379      0.426     -2.434      0.016      -1.882      -0.194
AdjR2: 0.2091


In [8]:
X= Fama_French_data[['Mkt-RF', 'SMB', 'HML']]
y = Fama_French_data['GM'] - Fama_French_data['RF']
X = sm.add_constant(X)
Fama_French_Model = sm.OLS(y, X).fit()
print(Fama_French_Model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.530
Model:                            OLS   Adj. R-squared:                  0.519
Method:                 Least Squares   F-statistic:                     47.72
Date:                Fri, 03 Apr 2026   Prob (F-statistic):           1.01e-20
Time:                        07:55:14   Log-Likelihood:                 180.34
No. Observations:                 131   AIC:                            -352.7
Df Residuals:                     127   BIC:                            -341.2
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0038      0.006     -0.681      0.4